# Información previa: *args* y *kwargs*

Antes de comenzar con los decoradores, es necesario entender qué significan las palabras *args* y *kwags*. Cuando no se quieren expresar de manera explícita los nombres de los parámetros de una función en Python, se utilizan estas dos variantes. Hay varios motivos para hacer esto:
- La lista de parámetros a indicar es variable. Imaginemos, por ejemplo, que queremos sumar un número indeterminado de números, matrices, etc.
- Hay muchos parámetros opcionales, y la mayoría de ellos los trata una función definida en una superclase.
- No sabemos qué datos nos van a pasar.
- Herencia: pasamos argumentos a una cierta superclase sin saber cuáles son.
- etc.

Veamos un par de ejemplos para ver cómo funcionan:

In [ ]:
def tomar_pedido(nombre_cliente, *platos, **detalles):
    # nombre_cliente es un parámetro al uso
    print(f"Pedido de {nombre_cliente}:")
    
    # platos es una lista, pero si no hay platos tenemos None
    if platos:
        print("Platos pedidos:")
        for plato in platos:
            print(f" - {plato}")
    else:
        print("No se han pedido platos.")
    
    # detalles es un diccionario {'nombre_del_argumento':'valor_del_argumento'}
    if detalles:
        print("Detalles adicionales:")
        for clave, valor in detalles.items():
            print(f" - {clave}: {valor}")
    else:
        print("Sin detalles adicionales.")

# Ejemplo de uso
tomar_pedido("Ana", "Pizza", "Ensalada", cubiertos=True, alergias="gluten")


In [ ]:

import numpy as np

def multiplicar_matrices(modo, *matrices):
    """
    Multiplica matrices de forma flexible.
    
    modo: 'pointwise' o 'matricial'
    *matrices: matrices a multiplicar
    """
    if len(matrices) < 2:
        raise ValueError("Se necesitan al menos dos matrices para multiplicar.")

    resultado = matrices[0]

    for matriz in matrices[1:]:
        if modo == 'pointwise':
            resultado = np.multiply(resultado, matriz)
        elif modo == 'matricial':
            resultado = np.matmul(resultado, matriz)
        else:
            raise ValueError("Modo no reconocido. Usa 'pointwise' o 'matricial'.")

    return resultado

# Ejemplo de uso
A = np.array([[1, 2], [3, 4]])
B = np.array([[2, 0], [1, 2]])
C = np.array([[5, 0], [-1, -2]])

# Multiplicación punto a punto
print(multiplicar_matrices('pointwise', A, B, C))

# Multiplicación matricial
print(multiplicar_matrices('matricial', A, B, C))


La función de *args* y *kwargs* se puede invertir: en vez de utilizarlos para leer argumentos, podemos usar esta notación para llamar a una lista de argumentos o a un diccionario de argumentos. Probemos con la función *print*.

In [ ]:
print("Esto", "es", "un", "print", "que", "deja", "dos", "espacios", sep="  ", end="\n")

In [ ]:
lista_args = ["Esto", "es", "un", "print", "que", "deja", "dos", "espacios"]
dict_kwargs = {'sep': "  ", 'end': "\n"}

print(*lista_args, **dict_kwargs)

# Decoradores

Ahora estamos preparados para ver decoradores. Un decorador permite añadir nuevas funcionalidades a una función genérica. Ya hemos visto algunos en clase: *property*, *setter*, *staticmethod*, *abstractmethod*... Todos tienen algo en común: no les importa lo que haya dentro de la función, solo extienden su comportamiento.

Un decorador en Python es una función que recibe un argumento (que es una función) y devuelve una función. Pongamos el ejemplo más típico de decorador: el medidor de tiempos.

In [ ]:
import time

# Nuestro decorador se llamará timer y le pasamos una función genérica
def timer(función_genérica):
    # wrapper es el nombre que se pone por convenio a la nueva función que vamos a crear
    def wrapper(*args, **kwargs):
        start_time = time.time()
        # Nos da igual lo que haga la función, le pasamos los argumentos que hemos recibido
        retorno = función_genérica(*args, **kwargs)
        end_time = time.time()
        
        print(f"La función {función_genérica.__name__} ha tardado {end_time - start_time} segundos.")
        
        return retorno
    
    # Devolvemos la nueva función
    return wrapper
        

In [ ]:
@timer
def tarea_lenta():
    time.sleep(1)
    print("Tarea completada")

tarea_lenta()

También se pueden pasar argumentos a los decoradores. Para ello, hay que encapsular el decorador anterior dentro de un decorador (sí, lo sé, esto parece Origen: funciones dentro de funciones).

In [ ]:
# Decorador que permite medir el tiempo que tarda una función ejecutándola n veces
def mide_tiempo(n):
    # Una vez que ya hemos recibido los parámetros, creamos el decorador como antes
    def timer_decorator(función_genérica):
        def wrapper(*args, **kwargs):
            start_time = time.time()
            
            for _ in range(n):
                retorno = función_genérica(*args, **kwargs)
                
            end_time = time.time()
                
            print(f"La función {función_genérica.__name__} tarda de media {(end_time - start_time)/n} segundos.")
                
            return retorno
        
        return wrapper
    return timer_decorator

In [ ]:
@mide_tiempo(10)
def tarea_lenta():
    time.sleep(1)
    print("Tarea completada")

tarea_lenta()